In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA = Path.cwd() / "olist_data"

orders = pd.read_csv(DATA / "orders.csv")
items = pd.read_csv(DATA / "items.csv")
reviews = pd.read_csv(DATA / "reviews.csv")

print("Orders:", len(orders))
print("Items:", len(items))
print("Reviews:", len(reviews))

display(orders.head())

In [ ]:
#cell 3
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

reviews["review_text"] = (
    reviews["review_text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

assert orders["order_id"].is_unique

assert set(items["order_id"]).issubset(
    set(orders["order_id"])
)

assert set(reviews["order_id"]).issubset(
    set(orders["order_id"])
)

print("Text reviews:", reviews["review_text"].ne("").sum())
print("Without text:", reviews["review_text"].eq("").sum())

In [ ]:
#cell 4
plt.figure(figsize=(6, 4))

plt.bar(
    ["Orders", "Items", "Reviews"],
    [len(orders), len(items), len(reviews)],
    color=["steelblue", "orange", "seagreen"]
)

plt.title("Dataset Size")
plt.ylabel("Rows")

plt.tight_layout()
plt.show()

In [ ]:
#cell 4
rating_counts = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(6, 4))

plt.bar(
    rating_counts.index,
    rating_counts.values,
    color="steelblue"
)

plt.xticks([1, 2, 3, 4, 5])
plt.title("Star Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Review Events")

plt.tight_layout()
plt.show()

In [ ]:
#cell 4
plt.figure(figsize=(6, 4))

plt.bar(
    ["With Text", "Without Text"],
    [
        reviews["review_text"].ne("").sum(),
        reviews["review_text"].eq("").sum()
    ],
    color=["seagreen", "orange"]
)

plt.title("Review Text Availability")
plt.ylabel("Review Events")

plt.tight_layout()
plt.show()

In [ ]:
#cell 5
sentiment_names = {
    1: "Very Negative",
    2: "Negative",
    3: "Neutral",
    4: "Positive",
    5: "Very Positive"
}

text_data = reviews[
    reviews["review_text"].ne("")
].copy()

text_data = text_data.sort_values(
    ["review_answer_timestamp", "review_id"]
).reset_index(drop=True)

text_data["sentiment_label"] = (
    text_data["review_score"].astype(int)
)

sentiment_counts = (
    text_data["sentiment_label"]
    .value_counts()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
)

plt.bar(
    list(sentiment_names.values()),
    sentiment_counts.values,
    color=["darkred", "tomato", "gray", "seagreen", "darkgreen"]
)

plt.title("Rating-Based Sentiment Labels")
plt.ylabel("Text Reviews")
plt.xticks(rotation=16)
plt.tight_layout()
plt.show()

In [ ]:
#cell 6 :create a sentiment model
def create_sentiment_model():

    model = make_pipeline(
        TfidfVectorizer(
            max_features=4000,
            ngram_range=(1, 2),
            min_df=2
        ),

        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )

    return model


 
split_position = int(len(text_data) * 0.80)

split_time = text_data.loc[
    split_position,
    "review_answer_timestamp"
]

sentiment_train = text_data[
    text_data["review_answer_timestamp"] < split_time
]

sentiment_test = text_data[
    text_data["review_answer_timestamp"] >= split_time
]

sentiment_model = create_sentiment_model()

sentiment_model.fit(
    sentiment_train["review_text"],
    sentiment_train["sentiment_label"]
)

sentiment_prediction = sentiment_model.predict(
    sentiment_test["review_text"]
)

print("Training reviews:", len(sentiment_train))
print("Testing reviews:", len(sentiment_test))

Training reviews: 3493
Testing reviews: 874


In [ ]:
#cell 7 : cionfusion matrix
sentiment_accuracy = accuracy_score(
    sentiment_test["sentiment_label"],
    sentiment_prediction
)

sentiment_f1 = f1_score(
    sentiment_test["sentiment_label"],
    sentiment_prediction,
    labels=[1, 2, 3, 4, 5],
    average="macro",
    zero_division=0
)

print("Accuracy:", round(sentiment_accuracy, 4))
print("Macro F1:", round(sentiment_f1, 4))

print(
    classification_report(
        sentiment_test["sentiment_label"],
        sentiment_prediction,
        labels=[1, 2, 3, 4, 5],
        target_names=list(sentiment_names.values()),
        zero_division=0
    )
)

matrix = confusion_matrix(
    sentiment_test["sentiment_label"],
    sentiment_prediction,
    labels=[1, 2, 3, 4, 5]
)

plt.figure(figsize=(6, 4))

sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=list(sentiment_names.values()),
    yticklabels=list(sentiment_names.values())
)

plt.title("Sentiment Proxy — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Rating-Based Label")
plt.tight_layout()
plt.show()

In [20]:
#cell 8
START_WEEK = pd.Timestamp("2017-01-02")

LAST_PARTIAL_WEEK = (
    orders["order_purchase_timestamp"]
    .max()
    .to_period("W-SUN")
    .start_time
)

weeks = pd.date_range(
    start=START_WEEK,
    end=LAST_PARTIAL_WEEK - pd.Timedelta(weeks=1),
    freq="W-MON"
)

reviews["review_week"] = (
    reviews["review_answer_timestamp"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

print("Complete calendar weeks:", len(weeks))

Complete calendar weeks: 59


In [ ]:
#cell 9 :sentiment for forecasting
reviews["predicted_sentiment"] = np.nan
reviews["agreement"] = np.nan

for week_number, week in enumerate(weeks):

    past_reviews = text_data[
        text_data["review_answer_timestamp"] < week
    ]

    current_reviews = reviews[
        (reviews["review_week"] == week)
        & reviews["review_text"].ne("")
    ]

 
    if len(past_reviews) < 100:
        continue

    if past_reviews["sentiment_label"].nunique() < 2:
        continue

    if len(current_reviews) == 0:
        continue

    weekly_sentiment_model = create_sentiment_model()

    weekly_sentiment_model.fit(
        past_reviews["review_text"],
        past_reviews["sentiment_label"]
    )

    prediction = weekly_sentiment_model.predict(
        current_reviews["review_text"]
    )

    reviews.loc[
        current_reviews.index,
        "predicted_sentiment"
    ] = prediction

 
    agreement = 1 - (
        np.abs(
            prediction
            - current_reviews["review_score"].to_numpy()
        ) / 4
    )

    reviews.loc[
        current_reviews.index,
        "agreement"
    ] = agreement

    if week_number % 10 == 0:
        print("Processed week:", week_number + 1)

print("Weekly sentiment predictions complete.")

In [ ]:
#cell 10 :historical sales order
dated_items = items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

dated_items["week_start"] = (
    dated_items["order_purchase_timestamp"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

historical = dated_items.groupby(
    ["product_category_name_english", "week_start"]
).agg(
    demand=("order_id", "nunique"),
    avg_price=("price", "mean"),
    active_products=("product_id", "nunique")
).reset_index()

historical = historical.rename(
    columns={"product_category_name_english": "category"}
)

categories = sorted(
    items["product_category_name_english"].unique()
)

 
grid = pd.MultiIndex.from_product(
    [categories, weeks],
    names=["category", "week_start"]
).to_frame(index=False)

weekly = grid.merge(
    historical,
    on=["category", "week_start"],
    how="left",
    validate="one_to_one"
)

weekly[["demand", "active_products"]] = (
    weekly[["demand", "active_products"]].fillna(0)
)

display(weekly.head())

,category,week_start,demand,avg_price,active_products
0,bed_bath_table,2017-01-02,1,68.900000,1
1,bed_bath_table,2017-01-09,5,59.900000,5
2,bed_bath_table,2017-01-16,10,100.100000,9
3,bed_bath_table,2017-01-23,16,74.182222,13
4,bed_bath_table,2017-01-30,34,92.839412,28


In [ ]:
#cell 11:Customer review features connet
order_categories = items[
    ["order_id", "product_category_name_english"]
].drop_duplicates()

order_categories = order_categories.rename(
    columns={"product_category_name_english": "category"}
)

review_events = reviews.merge(
    order_categories,
    on="order_id",
    how="inner",
    validate="many_to_many"
)


review_events["positive"] = np.where(
    review_events["predicted_sentiment"].notna(),
    (review_events["predicted_sentiment"] >= 4).astype(float),
    np.nan
)

review_events["negative"] = np.where(
    review_events["predicted_sentiment"].notna(),
    (review_events["predicted_sentiment"] <= 2).astype(float),
    np.nan
)

review_summary = review_events.groupby(
    ["category", "review_week"]
).agg(
    review_volume=("review_id", "size"),
    avg_rating=("review_score", "mean"),
    avg_sentiment=("predicted_sentiment", "mean"),
    positive_ratio=("positive", "mean"),
    negative_ratio=("negative", "mean"),
    sentiment_review_count=("predicted_sentiment", "count"),
    avg_agreement=("agreement", "mean")
).reset_index()

review_summary = review_summary.rename(
    columns={"review_week": "week_start"}
)

weekly = weekly.merge(
    review_summary,
    on=["category", "week_start"],
    how="left",
    validate="one_to_one"
)

weekly[["review_volume", "sentiment_review_count"]] = (
    weekly[["review_volume", "sentiment_review_count"]]
    .fillna(0)
)

weekly = weekly.sort_values(
    ["category", "week_start"]
).reset_index(drop=True)

display(weekly.head())

,category,week_start,demand,avg_price,active_products,review_volume,avg_rating,avg_sentiment,positive_ratio,negative_ratio,sentiment_review_count,avg_agreement
0,bed_bath_table,2017-01-02,1,68.900000,1,0.0,NaN,NaN,NaN,NaN,0.0,NaN
1,bed_bath_table,2017-01-09,5,59.900000,5,0.0,NaN,NaN,NaN,NaN,0.0,NaN
2,bed_bath_table,2017-01-16,10,100.100000,9,2.0,4.000000,NaN,NaN,NaN,0.0,NaN
3,bed_bath_table,2017-01-23,16,74.182222,13,6.0,4.666667,NaN,NaN,NaN,0.0,NaN
4,bed_bath_table,2017-01-30,34,92.839412,28,4.0,5.000000,NaN,NaN,NaN,0.0,NaN


In [ ]:
#cell 12:Historical demand ও review dynamics graphs
fig, axes = plt.subplots(3, 1, figsize=(10, 10))

for category in categories:

    category_data = weekly[
        weekly["category"] == category
    ]

    axes[0].plot(
        category_data["week_start"],
        category_data["demand"],
        label=category
    )

    axes[1].plot(
        category_data["week_start"],
        category_data["review_volume"],
        label=category
    )

    axes[2].plot(
        category_data["week_start"],
        category_data["avg_sentiment"],
        label=category
    )

axes[0].set_title("Historical Weekly Orders")
axes[0].set_ylabel("Unique Orders")

axes[1].set_title("Weekly Review Volume")
axes[1].set_ylabel("Review Events")

axes[2].set_title("Weekly Predicted Sentiment")
axes[2].set_ylabel("Average Sentiment")
axes[2].set_ylim(1, 5)

for ax in axes:
    ax.set_xlabel("Week")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
#cell 13:Previous orders, review changes ও target
for lag in [1, 2, 4]:

    weekly[f"demand_lag_{lag}"] = (
        weekly.groupby("category")["demand"]
        .shift(lag)
    )

weekly["rolling_avg_4w"] = (
    weekly.groupby("category")["demand"]
    .transform(
        lambda values: values.rolling(
            window=4,
            min_periods=4
        ).mean()
    )
)

 
for column in [
    "review_volume",
    "avg_rating",
    "avg_sentiment"
]:

    weekly[column + "_change"] = (
        weekly.groupby("category")[column].diff()
    )

 
weekly["target_next_week"] = (
    weekly.groupby("category")["demand"].shift(-1)
)

weekly["target_week"] = (
    weekly["week_start"] + pd.Timedelta(weeks=1)
)

category_codes = {}

for number, category in enumerate(categories):
    category_codes[category] = number

weekly["category_code"] = (
    weekly["category"].map(category_codes)
)

model_data = weekly.dropna(
    subset=[
        "demand_lag_4",
        "rolling_avg_4w",
        "target_next_week"
    ]
).copy()

print("Forecasting rows:", len(model_data))

display(
    model_data[
        [
            "category",
            "week_start",
            "demand",
            "demand_lag_1",
            "rolling_avg_4w",
            "review_volume",
            "avg_sentiment",
            "target_next_week"
        ]
    ].head()
)

Forecasting rows: 108


,category,week_start,demand,demand_lag_1,rolling_avg_4w,review_volume,avg_sentiment,target_next_week
4,bed_bath_table,2017-01-30,34,16.0,16.25,4.0,NaN,43.0
5,bed_bath_table,2017-02-06,43,34.0,25.75,16.0,NaN,39.0
6,bed_bath_table,2017-02-13,39,43.0,33.00,32.0,NaN,32.0
7,bed_bath_table,2017-02-20,32,39.0,37.00,39.0,NaN,48.0
8,bed_bath_table,2017-02-27,48,32.0,40.50,29.0,3.578947,59.0


In [28]:
#cell 14:feature determine for two model
# Model 1: Historical Only
historical_features = [
    "category_code",
    "demand",
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_4",
    "rolling_avg_4w",
    "avg_price",
    "active_products"
]

# Model 2: Historical + Reviews
review_features = [
    "review_volume",
    "avg_rating",
    "avg_sentiment",
    "positive_ratio",
    "negative_ratio",
    "sentiment_review_count",
    "avg_agreement",
    "review_volume_change",
    "avg_rating_change",
    "avg_sentiment_change"
]

combined_features = historical_features + review_features

feature_sets = {
    "Historical Only": historical_features,
    "Historical + Reviews": combined_features
}

print("Historical features:", len(historical_features))
print("Combined features:", len(combined_features))

Historical features: 8
Combined features: 18


In [30]:
#cell 15:Time-based train/validation/test split
all_target_weeks = sorted(
    model_data["target_week"].unique()
)

number_of_weeks = len(all_target_weeks)

train_end = int(number_of_weeks * 0.70)
validation_end = int(number_of_weeks * 0.85)

train_weeks = all_target_weeks[:train_end]

validation_weeks = all_target_weeks[
    train_end:validation_end
]

test_weeks = all_target_weeks[validation_end:]

train_data = model_data[
    model_data["target_week"].isin(train_weeks)
].copy()

validation_data = model_data[
    model_data["target_week"].isin(validation_weeks)
].copy()

test_data = model_data[
    model_data["target_week"].isin(test_weeks)
].copy()

assert (
    train_data["target_week"].max()
    < validation_data["target_week"].min()
)

assert (
    validation_data["target_week"].max()
    < test_data["target_week"].min()
)

print("Train rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Test rows:", len(test_data))

Train rows: 74
Validation rows: 16
Test rows: 18


In [ ]:
#cell 16:Result calculation function
def calculate_results(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

 
    nonzero = actual != 0

    if nonzero.any():

        mape = np.mean(
            np.abs(
                (actual[nonzero] - predicted[nonzero])
                / actual[nonzero]
            )
        ) * 100

    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R2": r2
    }

In [ ]:
#cell 17:XGboost model train
models = {}
validation_rows = []

for name, columns in feature_sets.items():

    model = make_pipeline(
        SimpleImputer(strategy="median"),

        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            reg_lambda=5,
            random_state=42,
            n_jobs=2
        )
    )

    model.fit(
        train_data[columns],
        train_data["target_next_week"]
    )

    prediction = model.predict(
        validation_data[columns]
    )

  
    prediction = np.maximum(prediction, 0)

    models[name] = model

    validation_rows.append({
        "Model": name,
        **calculate_results(
            validation_data["target_next_week"],
            prediction
        )
    })

validation_results = pd.DataFrame(validation_rows)

best_model_name = (
    validation_results
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

display(validation_results.round(4))

print("Selected by validation RMSE:", best_model_name)

,Model,MAE,RMSE,MAPE (%),R2
0,Historical Only,45.6588,81.4334,24.4506,-0.2629
1,Historical + Reviews,46.0168,80.2001,25.0113,-0.2249


Selected by validation RMSE: Historical + Reviews


In [ ]:
#cell 18:Test result evaluation
test_rows = []
test_predictions = {}

for name, columns in feature_sets.items():

    prediction = models[name].predict(
        test_data[columns]
    )

    prediction = np.maximum(prediction, 0)

    test_predictions[name] = prediction

    test_rows.append({
        "Model": name,
        **calculate_results(
            test_data["target_next_week"],
            prediction
        )
    })

test_results = pd.DataFrame(test_rows)

display(test_results.round(4))

print("Validation-selected model:", best_model_name)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

test_results.set_index("Model")["MAE"].plot(
    kind="bar",
    ax=axes[0],
    color=["steelblue", "seagreen"]
)

test_results.set_index("Model")["RMSE"].plot(
    kind="bar",
    ax=axes[1],
    color=["steelblue", "seagreen"]
)

axes[0].set_title("Historical vs Historical + Reviews: MAE")
axes[1].set_title("Historical vs Historical + Reviews: RMSE")

for ax in axes:
    ax.set_ylabel("Error — Lower is Better")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
#cell 19:add review for improvement
historical_result = test_results[
    test_results["Model"] == "Historical Only"
].iloc[0]

combined_result = test_results[
    test_results["Model"] == "Historical + Reviews"
].iloc[0]

for metric in ["MAE", "RMSE"]:

    old_error = historical_result[metric]
    new_error = combined_result[metric]

    if old_error > 0:

        improvement = (
            (old_error - new_error) / old_error
        ) * 100

        print(
            metric,
            "improvement:",
            round(improvement, 2),
            "%"
        )

    else:
        print(metric, "baseline error is zero.")

print("\nPositive % = error decreased.")
print("Negative % = error increased.")

In [ ]:
#20 :actual vs predicted model
prediction_table = test_data[
    [
        "category",
        "target_week",
        "target_next_week"
    ]
].copy()

prediction_table = prediction_table.rename(
    columns={"target_next_week": "actual_orders"}
)

prediction_table["historical_prediction"] = (
    test_predictions["Historical Only"]
)

prediction_table["review_prediction"] = (
    test_predictions["Historical + Reviews"]
)

fig, axes = plt.subplots(
    len(categories),
    1,
    figsize=(12, 9),
    squeeze=False
)

for position, category in enumerate(categories):

    result = prediction_table[
        prediction_table["category"] == category
    ].sort_values("target_week")

    ax = axes[position, 0]

    ax.plot(
        result["target_week"],
        result["actual_orders"],
        marker="o",
        label="Actual Orders",
        color="black"
    )

    ax.plot(
        result["target_week"],
        result["historical_prediction"],
        marker="s",
        label="Historical Only"
    )

    ax.plot(
        result["target_week"],
        result["review_prediction"],
        marker="^",
        label="Historical + Reviews"
    )

    ax.set_title(category)
    ax.set_xlabel("Target Week")
    ax.set_ylabel("Orders")
    ax.legend()

plt.tight_layout()
plt.show()

display(prediction_table.round(2))

In [38]:
#cell 22:save
OUTPUT = Path.cwd() / "project_results"
OUTPUT.mkdir(exist_ok=True)

weekly.to_csv(
    OUTPUT / "weekly_features.csv",
    index=False
)

validation_results.to_csv(
    OUTPUT / "validation_results.csv",
    index=False
)

test_results.to_csv(
    OUTPUT / "test_results.csv",
    index=False
)

prediction_table.to_csv(
    OUTPUT / "actual_vs_predictions.csv",
    index=False
)

print("Results saved in:", OUTPUT.resolve())

Results saved in: C:\Users\MSI MAG\Downloads\DM_ali\project_results


In [40]:
display(test_results.round(4))

,Model,MAE,RMSE,MAPE (%),R2
0,Historical Only,43.9028,48.5848,33.2634,-1.0924
1,Historical + Reviews,43.0038,47.8477,32.1428,-1.0294


In [ ]:
display(prediction_table.round(2))